# Pharmasense — Anomaly Detector
**Detects outbreak spikes using Isolation Forest + Z-score on rolling baseline.**

Covers all diseases. Triggers alerts when cases exceed 2.5σ above seasonal baseline.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhi-dev99/agtechathon/blob/main/ml/notebooks/06_train_anomaly_detector.ipynb)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn'], check=True)
print('Done.')

In [ ]:
GITHUB_RAW = 'https://raw.githubusercontent.com/abhi-dev99/agtechathon/main/ml/data'
DRIVE_OUTPUT = '/content/drive/MyDrive/pharmasense_models/'
import os
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
import os
from google.colab import files
REQUIRED_FILES = ['malaria_weekly.csv.gz', 'epiclim_clean.csv']
missing = [f for f in REQUIRED_FILES if not os.path.exists(f'/content/{f}')]
if missing:
    print('GitHub repository is private. Please upload the following files from your local ml/data/ folder:\n')
    for f in missing:
        print(f'  - {f}')
    print('\n')
    uploaded = files.upload()
    missing = [f for f in REQUIRED_FILES if not os.path.exists(f'/content/{f}')]
    if missing:
        raise FileNotFoundError(f'Still missing: {missing}')
print('All data files are present.')


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import seaborn as sns

# Load malaria
mal = pd.read_csv('/content/malaria_weekly.csv.gz', parse_dates=['week_start_date'])
mal_agg = mal.groupby('week_start_date')['cases'].sum().reset_index()
mal_agg['disease'] = 'malaria'
mal_agg.columns = ['week','cases','disease']

# Load Epiclim
epi = pd.read_csv('/content/epiclim_clean.csv')
epi['week_of_outbreak'] = pd.to_datetime(epi[['year','mon','day']].rename(columns={'mon':'month'}))
epi_agg = epi.groupby(['week_of_outbreak','Disease'])['Cases'].sum().reset_index()
epi_agg.columns = ['week','disease','cases']
epi_agg['disease'] = epi_agg['disease'].str.lower().str.replace(' ','_')

df = pd.concat([mal_agg, epi_agg], ignore_index=True)
df = df.sort_values(['disease','week']).reset_index(drop=True)
print(f'Total records: {len(df):,} | Diseases: {df.disease.unique()}')

In [ ]:
# ── Feature engineering ───────────────────────────────────────────────────────
df['log_cases'] = np.log1p(df['cases'])
df['month'] = df['week'].dt.month
df['year']  = df['week'].dt.year

# Rolling statistics per disease (4-week, 13-week windows)
records = []
for disease, grp in df.groupby('disease'):
    grp = grp.sort_values('week').copy()
    grp['roll4_mean']  = grp['log_cases'].rolling(4, min_periods=2).mean()
    grp['roll4_std']   = grp['log_cases'].rolling(4, min_periods=2).std().fillna(0)
    grp['roll13_mean'] = grp['log_cases'].rolling(13, min_periods=4).mean()
    grp['roll13_std']  = grp['log_cases'].rolling(13, min_periods=4).std().fillna(0)
    grp['z_score_4']   = (grp['log_cases'] - grp['roll4_mean']) / (grp['roll4_std'] + 1e-6)
    grp['z_score_13']  = (grp['log_cases'] - grp['roll13_mean']) / (grp['roll13_std'] + 1e-6)
    # Ground truth anomaly: z > 2.5 relative to 13-week baseline
    grp['is_anomaly_gt'] = (grp['z_score_13'] > 2.5).astype(int)
    records.append(grp)

df = pd.concat(records).dropna(subset=['roll4_mean']).reset_index(drop=True)
print(f'Features ready: {len(df):,} records')
print(f'Known anomalies (z>2.5): {df.is_anomaly_gt.sum()} ({df.is_anomaly_gt.mean()*100:.1f}%)')

In [ ]:
# ── Train Isolation Forest ────────────────────────────────────────────────────
FEATURES = ['log_cases','roll4_mean','roll4_std','roll13_mean','roll13_std',
            'z_score_4','z_score_13','month']

X = df[FEATURES].fillna(0).values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

iso = IsolationForest(n_estimators=200, contamination=0.05, random_state=42, n_jobs=-1)
iso.fit(X_scaled)
df['iso_score']   = iso.decision_function(X_scaled)  # more negative = more anomalous
df['iso_anomaly'] = (iso.predict(X_scaled) == -1).astype(int)

print('Isolation Forest trained.')
print(f'  IF anomalies detected: {df.iso_anomaly.sum()} ({df.iso_anomaly.mean()*100:.1f}%)')
print()
# Evaluate against z-score ground truth
print('Classification vs z-score ground truth:')
print(classification_report(df['is_anomaly_gt'], df['iso_anomaly'],
                             target_names=['Normal','Anomaly']))

In [ ]:
# ── Plot anomaly timeline per disease ────────────────────────────────────────
import matplotlib; matplotlib.use('Agg')
diseases = df['disease'].unique()
n = len(diseases)
fig, axes = plt.subplots(n, 1, figsize=(16, 4*n))
if n == 1: axes = [axes]

for ax, disease in zip(axes, sorted(diseases)):
    sub = df[df['disease']==disease].sort_values('week')
    ax.fill_between(sub['week'], 0, np.expm1(sub['log_cases']),
                    alpha=0.4, color='#3B82F6', label='Cases')
    anomalies = sub[sub['iso_anomaly']==1]
    ax.scatter(anomalies['week'], np.expm1(anomalies['log_cases']),
               color='#DC2626', zorder=5, s=40, label='Anomaly')
    ax.set_title(f'{disease.replace("_"," ").title()} — Outbreak Detection', fontweight='bold')
    ax.set_ylabel('Cases/week')
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'anomaly_detection_all_diseases.png', dpi=150)
plt.show()
print('Timeline plots saved.')

In [ ]:
# ── Anomaly score distribution ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['iso_score'], bins=60, color='#3B82F6', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='red', linestyle='--', label='Decision boundary')
axes[0].set_title('Isolation Forest Anomaly Score Distribution')
axes[0].set_xlabel('Anomaly Score (more negative = more anomalous)')
axes[0].legend()

sns.boxplot(data=df, x='disease', y='iso_score', ax=axes[1], palette='Set2')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Anomaly Scores by Disease')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig(DRIVE_OUTPUT + 'anomaly_score_distribution.png', dpi=150)
plt.show()

In [ ]:
# ── Save ────────────────────────────────────────────────────────────────────
import pickle
from datetime import datetime
artifact = {'model': iso, 'scaler': scaler, 'features': FEATURES,
            'z_threshold': 2.5, 'contamination': 0.05,
            'trained_at': datetime.now().isoformat(),
            'anomaly_rate': float(df.iso_anomaly.mean())}
out_path = DRIVE_OUTPUT + 'anomaly_detector.pkl'
with open(out_path, 'wb') as f:
    pickle.dump(artifact, f)
from google.colab import files
files.download(out_path)
print(f'Saved: {out_path}')